In [8]:
from pathlib import Path
import sys

# 1. Asegurar que la carpeta 'src' esté en las rutas de importación de Python
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
src_dir = PROJECT_ROOT / "src"
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

# 2. Ahora sí se puede importar directamente
from inf8239_u01.data import download_csv

# 3. Descarga reproducible oficial desde UCI
URL = "https://archive.ics.uci.edu/static/public/601/ai4i+2020+predictive+maintenance+dataset.zip"
dest_path = PROJECT_ROOT / "data" / "raw" / "dataset.csv"

path = download_csv(URL, destination=str(dest_path))
print("Descarga completada y verificada en:", path)

Descarga completada y verificada en: /workspaces/INF8239_U01/data/raw/dataset.csv


In [10]:
import pandas as pd

PROJECT_ROOT = (Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd())
csv_path = PROJECT_ROOT / "data" / "raw" / "dataset.csv"

df = pd.read_csv(csv_path)

print(df.shape)
print(df.dtypes)
print(df.head())
print(df.tail())
assert not df.empty

(10000, 14)
UDI                          int64
Product ID                     str
Type                           str
Air temperature [K]        float64
Process temperature [K]    float64
Rotational speed [rpm]       int64
Torque [Nm]                float64
Tool wear [min]              int64
Machine failure              int64
TWF                          int64
HDF                          int64
PWF                          int64
OSF                          int64
RNF                          int64
dtype: object
   UDI Product ID Type  Air temperature [K]  Process temperature [K]  \
0    1     M14860    M                298.1                    308.6   
1    2     L47181    L                298.2                    308.7   
2    3     L47182    L                298.1                    308.5   
3    4     L47183    L                298.2                    308.6   
4    5     L47184    L                298.2                    308.7   

   Rotational speed [rpm]  Torque [Nm]  Tool wear [

In [11]:
import pandas as pd

audit = pd.DataFrame(
    {
        "tipo": df.dtypes.astype(str),
        "ausentes": df.isna().sum(),
        "porcentaje_ausente": (df.isna().mean() * 100).round(2),
        "unicos": df.nunique(dropna=False),
    }
).sort_values("porcentaje_ausente", ascending=False)

print("Duplicados:", df.duplicated().sum())
display(audit)

Duplicados: 0


,tipo,ausentes,porcentaje_ausente,unicos
UDI,int64,0,0.0,10000
Product ID,str,0,0.0,10000
Type,str,0,0.0,3
Air temperature [K],float64,0,0.0,93
Process temperature [K],float64,0,0.0,82
Rotational speed [rpm],int64,0,0.0,941
Torque [Nm],float64,0,0.0,577
Tool wear [min],int64,0,0.0,246
Machine failure,int64,0,0.0,2
TWF,int64,0,0.0,2


In [12]:
TARGET = "Machine failure"

# Lista de columnas a excluir del conjunto de entrenamiento:
DROP_COLUMNS = [
    # 1. IDENTIFICADORES TÉCNICOS (Razón semántica):
    # 'UDI' y 'Product ID' tienen cardinalidad única (10,000 valores distintos).
    # No aportan información física del proceso y provocarían sobreajuste
    # si el modelo intenta memorizar el identificador.
    "UDI",
    "Product ID",
    # 2. MODOS DE FALLA ESPECÍFICOS (Razón de disponibilidad / Data Leakage):
    # TWF (Tool Wear Failure), HDF (Heat Dissipation Failure), PWF (Power Failure),
    # OSF (Overstrain Failure) y RNF (Random Failure) detallan la causa de la avería.
    # Estas variables solo se conocen DESPUÉS de que la máquina falló.
    # Si las dejamos, el modelo aprendería con información del futuro que no existe
    # al momento de hacer mantenimiento predictivo en tiempo real.
    "TWF",
    "HDF",
    "PWF",
    "OSF",
    "RNF",
]

assert TARGET in df.columns
X = df.drop(columns=[TARGET] + DROP_COLUMNS)
y = df[TARGET]

print("Distribución de clases en el target:")
print(y.value_counts(dropna=False))

assert y.notna().all()
assert y.nunique() >= 2

print("\nColumnas predictoras retenidas en X:")
print(list(X.columns))

Distribución de clases en el target:
Machine failure
0    9661
1     339
Name: count, dtype: int64

Columnas predictoras retenidas en X:
['Type', 'Air temperature [K]', 'Process temperature [K]', 'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]']


In [13]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

num_cols = X.select_dtypes(include="number").columns.tolist()
cat_cols = X.select_dtypes(exclude="number").columns.tolist()

num_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scale", StandardScaler()),
    ])

cat_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ])

preprocess = ColumnTransformer([
        ("num", num_pipe, num_cols),
        ("cat", cat_pipe, cat_cols),
    ])

print(len(num_cols), len(cat_cols))

5 1


In [14]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import classification_report, f1_score
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC

Xtr, Xte, ytr, yte = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
dummy = Pipeline([
        ("prep", preprocess),
        ("model", DummyClassifier(strategy="most_frequent")),
    ])
svm = Pipeline([
        ("prep", preprocess),
        ("model",SVC(C=1,gamma="scale",probability=True,random_state=42))
    ])
    
for name, model in {"dummy": dummy, "svm": svm}.items():
    model.fit(Xtr, ytr)
    pred = model.predict(Xte)
    print(name, f1_score(yte, pred, average="macro"))
print(classification_report(yte, svm.predict(Xte)))

dummy 0.491353001017294


/workspaces/INF8239_U01/.venv/lib/python3.14/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


svm 0.6595165134490977
              precision    recall  f1-score   support

           0       0.97      1.00      0.99      1932
           1       0.88      0.21      0.33        68

    accuracy                           0.97      2000
   macro avg       0.92      0.60      0.66      2000
weighted avg       0.97      0.97      0.96      2000



> **Conclusión Técnica del Experimento y Auditoría de Datos**
>
> La selección y auditoría rigurosa del conjunto de datos **AI4I 2020 Predictive Maintenance** permitió sentar una base metodológica sólida y reproducible para la toma de decisiones en entornos de manufactura industrial. A través de la auditoría inicial se constató la integridad del archivo sin la presencia de valores ausentes (*missing values*) ni registros duplicados a lo largo de sus 10,000 instancias. No obstante, el desafío central del problema reside en dos dimensiones fundamentales: el severo desbalance de clases y la presencia de variables causales con potencial de fuga de información (*data leakage*).
>
> La variable dependiente `Machine failure` presenta apenas una tasa de incidencia del 3.39% frente a más de un 96% de estados operativos normales. Bajo este contexto, el uso de la exactitud (*accuracy*) como métrica rectora resulta metodológicamente inválido, tal como evidenció el modelo base trivial (*DummyClassifier*). Este clasificador ingenuo, al limitarse a predecir sistemáticamente la clase mayoritaria, obtiene un accuracy superior al 96% pero un **F1-macro inferior a 0.50**, fracasando rotundamente al predecir 0% de los eventos reales de fallo. Por tal motivo, se adoptó el **F1-macro** y el análisis de la matriz de confusión, donde el **falso negativo** representa el error más crítico: ignorar una falla incipiente conlleva roturas catastróficas, tiempos de inactividad imprevistos y costos operativos desproporcionados para la planta.
>
> Asimismo, la fase de saneamiento de variables fue determinante. La exclusión analítica de los identificadores (`UDI` y `Product ID`) evitó sobreajustes espurios por memorización de secuencias, mientras que la eliminación deliberada de los cinco modos de falla (`TWF`, `HDF`, `PWF`, `OSF`, `RNF`) blindó el flujo ante fugas del futuro. Al ser etiquetas que solo se constatan con posterioridad a la interrupción funcional de la máquina, incluirlas en el conjunto de entrenamiento habría provocado un modelo artificialmente perfecto pero inútil en un esquema de telemetría en tiempo real.
>
> Finalmente, la integración del pipeline con `ColumnTransformer` (escalando variables continuas como temperatura, torque y velocidad de giro, y codificando la variante cualitativa de producto) junto a una máquina de soporte vectorial (SVM) con núcleo RBF, demostró la viabilidad de capturar dependencias no lineales complejas entre las tensiones mecánicas y los gradientes térmicos. El establecimiento de pruebas automatizadas mediante `pytest` garantiza que las restricciones dimensionales y semánticas del contrato de datos perduren de manera confiable a lo largo de las iteraciones analíticas posteriores.